# 🚀 High-Level Agentic AI: Persistent Memory, ChromaDB & Modular Plugins

In this notebook, we build an enterprise-level modular AI Agent using:
1. **Connectors**: Database (Pandas/SQL), **ChromaDB Persistent Vector Store** (Anti-Hallucination Grounding & Long-Term User Memory), REST APIs, Document Engine (`fpdf2` & `pypdf`).
2. **Plugins & Tool Registry**: Modular domain tools (`CatalogPlugin`, `RAGSupportPlugin`, `FinancePlugin`, `InvoicePlugin`, `MemoryPlugin`).
3. **Persistent Memory & Checkpointing**: Thread-based conversation persistence (`thread_id`) and cross-session user memories.
4. **High-Level Agent**: With a simple `ask(question, user_id, thread_id)` interface.

## 1. Setup & Environment

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Ensure package is found in path
sys.path.insert(0, "src")

load_dotenv("src/agentic_ai/.env")
load_dotenv(".env")

print("Environment configured!")
print("GROQ_API_KEY Available:", bool(os.getenv("GROQ_API_KEY")))
print("GOOGLE_API_KEY Available:", bool(os.getenv("GOOGLE_API_KEY")))

## 2. ChromaDB Persistent Vector & Memory Connector

In [ ]:
from agentic_ai.connectors import ChromaMemoryConnector

chroma_conn = ChromaMemoryConnector(persist_dir="data/chroma_db")
chroma_conn.connect()

print(f"ChromaDB Connected at: {chroma_conn.persist_dir}")
print(f"Ground-Truth Knowledge Collection count: {chroma_conn.kb_collection.count()}")
print(f"User Long-Term Memory Collection count: {chroma_conn.memory_collection.count()}")

## 3. Registering All Domain Plugins (Including MemoryPlugin)

In [ ]:
from agentic_ai.plugins import PluginRegistry, CatalogPlugin, RAGSupportPlugin, FinancePlugin, InvoicePlugin, MemoryPlugin

registry = PluginRegistry()
registry.register(CatalogPlugin())
registry.register(RAGSupportPlugin())
registry.register(FinancePlugin())
registry.register(InvoicePlugin())
registry.register(MemoryPlugin(chroma_connector=chroma_conn))

print("=== Registered Work Plugins ===")
for summary in registry.get_summary():
    print(f"📦 {summary['plugin']} ({summary['tool_count']} tools): {', '.join(summary['tools'])}")

## 4. Initialize High-Level Agent with Persistent Checkpointer

In [ ]:
from agentic_ai.core import HighLevelAgent, get_llm

llm = get_llm(provider="groq", model_name="openai/gpt-oss-120b", temperature=0.0)
agent = HighLevelAgent(llm=llm, registry=registry)

def ask(question: str, user_id: str = "user_123", thread_id: str = "session_1"):
    print(f"\n👤 User [{user_id} | Thread: {thread_id}]: {question}")
    answer = agent.ask(question=question, user_id=user_id, thread_id=thread_id)
    print(f"🤖 Assistant:\n{answer}\n" + "-"*65)

print("Agent ready with thread persistence and ChromaDB grounding!")

## 5. Storing & Recalling Long-Term User Memories
Let's store a customer's setup & budget constraint in ChromaDB:

In [ ]:
# Save memory for customer 'sarah_connor'
save_result = chroma_conn.save_user_memory(
    user_id="sarah_connor",
    memory_fact="User is a video editor using a MacBook Air M3 and needs a color-accurate 4K monitor under $600 with 90W USB-C charging."
)
print("Saved Memory:", save_result)

### Testing Personalized, Grounded Query (Session A)
Notice how the agent automatically recalls Sarah's laptop and budget without hallucinating:

In [ ]:
ask("What monitor do you recommend for my workstation setup and budget?", user_id="sarah_connor", thread_id="sarah_thread_1")

### Testing Checkpointer Thread Persistence (Follow-up Turn)
The checkpointer retains context from the previous turn in the conversation thread:

In [ ]:
ask("What is the exact warranty on that monitor and what discounts can I apply?", user_id="sarah_connor", thread_id="sarah_thread_1")

## 6. Anti-Hallucination Grounding Check with ChromaDB
Check exact factual documentation from the persistent ChromaDB collection:

In [ ]:
verified_excerpts = chroma_conn.verify_ground_truth("Dell UltraSharp 27 4K IPS Black 90W USB-C")
for doc in verified_excerpts:
    print("\n--- Ground Truth Document ---")
    print(doc["content"][:250], "...")